# 第10章 蒙特卡洛树搜索

## 目录
1. [环境搭建](#1-环境搭建)
2. [MCTS算法](#2-MCTS算法)
3. [UCB选择策略](#3-UCB选择策略)
4. [AlphaGo中的应用](#4-AlphaGo中的应用)
5. [编程题](#5-编程题)

---

## 1. 环境搭建

In [ ]:
!pip install numpy matplotlib -q
import numpy as np
import matplotlib.pyplot as plt
import random
from math import sqrt, log
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print("环境搭建完成!")

---

## 2. MCTS算法

**蒙特卡洛树搜索（MCTS）**是一种基于树的搜索算法：
- 选择(Selection)：沿UCB准则选择子节点
- 扩展(Expansion)：添加新子节点
- 模拟(Simulation)：随机走子到终止
- 回溯(Backup)：更新路径上所有节点的价值

In [ ]:
class TicTacToeEnv:
    """井字棋环境"""
    def __init__(self):
        self.board = np.zeros(9)
        self.current = 1
    def reset(self): self.board = np.zeros(9); self.current = 1; return self.board.copy()
    def step(self, action):
        if self.board[action] != 0: return self.board.copy(), -10, True
        self.board[action] = self.current
        done, winner = self.check_winner()
        reward = 1 if winner == self.current else (-1 if winner != 0 else 0)
        self.current *= -1
        return self.board.copy(), reward, done
    def check_winner(self):
        wins = [(0,1,2), (3,4,5), (6,7,8), (0,3,6), (1,4,7), (2,5,8), (0,4,8), (2,4,6)]
        for a, b, c in wins:
            if self.board[a] != 0 and self.board[a] == self.board[b] == self.board[c]:
                return True, self.board[a]
        if np.all(self.board != 0): return True, 0
        return False, 0
    def get_valid_actions(self):
        return [i for i in range(9) if self.board[i] == 0]

class MCTSNode:
    """MCTS节点"""
    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action
        self.children = {}
        self.visits = 0
        self.value = 0.0
    
    def ucb(self, c=1.414):
        if self.visits == 0: return float('inf')
        exploit = self.value / self.visits
        explore = c * sqrt(log(self.parent.visits + 1) / self.visits)
        return exploit + explore
    
    def select_child(self):
        if not self.children: return None
        return max(self.children.values(), key=lambda x: x.ucb())

class MCTS:
    """MCTS算法"""
    def __init__(self, c=1.414, n_iterations=1000):
        self.c = c
        self.n_iterations = n_iterations
    
    def search(self, env):
        root = MCTSNode(env.reset())
        for _ in range(self.n_iterations):
            node = root
            env_copy = TicTacToeEnv()
            env_copy.board = node.state.copy()
            
            while node.children and not env_copy.check_winner()[0]:
                node = node.select_child()
                env_copy.board = node.state.copy()
            
            if not env_copy.check_winner()[0]:
                valid = env_copy.get_valid_actions()
                if valid:
                    action = random.choice(valid)
                    env_copy.board[action] = env_copy.current
                    node.children[action] = MCTSNode(env_copy.board.copy(), node, action)
                    node = node.children[action]
            
            reward = self.simulate(env_copy)
            
            while node:
                node.visits += 1
                node.value += reward
                node = node.parent
        
        return root.select_child().action if root.children else 0
    
    def simulate(self, env):
        done = False
        while not done:
            valid = env.get_valid_actions()
            if not valid: break
            action = random.choice(valid)
            _, reward, done = env.step(action)
        return reward

def mcts_demo():
    mcts = MCTS(c=1.414, n_iterations=500)
    wins = 0
    for _ in range(20):
        env = TicTacToeEnv()
        action = mcts.search(env)
        _, reward, done = env.step(action)
        if reward == 1: wins += 1
    print(f"MCTS胜率: {wins/20:.1%}")

mcts_demo()

---

## 3. UCB选择策略

In [ ]:
def compare_ucb():
    for c in [0.5, 1.0, 1.414, 2.0]:
        mcts = MCTS(c=c, n_iterations=200)
        wins = 0
        for _ in range(20):
            env = TicTacToeEnv()
            action = mcts.search(env)
            _, reward, _ = env.step(action)
            if reward == 1: wins += 1
        print(f"c={c}: 胜率={wins/20:.1%}")

compare_ucb()

---

## 4. AlphaGo中的应用

In [ ]:
class AlphaGoMCTS:
    """AlphaGo风格MCTS"""
    def __init__(self, c=1.414, n_iter=1000):
        self.c = c
        self.n_iter = n_iter
    
    def ucb_with_prior(self, node, prior=1.0):
        if node.visits == 0: return float('inf')
        exploit = node.value / node.visits
        explore = self.c * prior * sqrt(node.parent.visits) / (node.visits + 1)
        return exploit + explore

print("AlphaGo MCTS已实现")

---

## 5. 编程题

### 编程题1：Rollout算法

In [ ]:
class GridWorldEnv:
    """网格世界环境"""
    def __init__(self, size=4, goal=None, trap=None):
        self.size = size
        self.goal = goal if goal else (size-1, size-1)
        self.trap = trap if trap else [(size-2, size-1), (size-1, size-2)]
        self.start = (0, 0)
        self.reset()
    
    def reset(self):
        self.pos = self.start
        return self.pos
    
    def step(self, action):
        moves = [(0, -1), (0, 1), (-1, 0), (1, 0)]
        new_pos = (self.pos[0] + moves[action][0], self.pos[1] + moves[action][1])
        new_pos = (max(0, min(self.size-1, new_pos[0])), max(0, min(self.size-1, new_pos[1])))
        self.pos = new_pos
        
        if self.pos == self.goal: return self.pos, 1.0, True
        if self.pos in self.trap: return self.pos, -1.0, True
        return self.pos, -0.01, False
    
    def get_valid_actions(self): return [0, 1, 2, 3]

def random_rollout(env, max_steps=50):
    env_copy = GridWorldEnv(env.size, env.goal, env.trap)
    env_copy.pos = env.pos
    total_reward = 0
    for _ in range(max_steps):
        valid = env_copy.get_valid_actions()
        action = random.choice(valid)
        _, reward, done = env_copy.step(action)
        total_reward += reward
        if done: break
    return total_reward

def test_rollout():
    env = GridWorldEnv(size=4)
    rewards = []
    for _ in range(100):
        env.reset()
        total = 0
        for __ in range(20):
            action = random.choice(env.get_valid_actions())
            _, r, done = env.step(action)
            total += r
            if done: break
        rewards.append(total)
    print(f"平均回报: {np.mean(rewards):.3f}")
    return rewards

test_rollout()

### 编程题2：MCTS井字棋

In [ ]:
def test_mcts_tictactoe():
    """测试MCTS井字棋"""
    mcts = MCTS(n_iterations=500)
    wins = 0
    for _ in range(20):
        env = TicTacToeEnv()
        action = mcts.search(env)
        _, reward, done = env.step(action)
        if reward == 1: wins += 1
    print(f"MCTS胜率: {wins/20:.1%}")
    return wins

test_mcts_tictactoe()

### 编程题3：动作剪枝Rollout

In [ ]:
def pruned_rollout(env, prune_ratio=0.5, max_steps=30):
    valid = env.get_valid_actions()
    n_keep = max(1, int(len(valid) * prune_ratio))
    scores = {}
    for a in valid:
        tmp_env = GridWorldEnv(env.size, env.goal, env.trap)
        tmp_env.pos = env.pos
        _, r, _ = tmp_env.step(a)
        dist_goal = abs(tmp_env.pos[0] - env.goal[0]) + abs(tmp_env.pos[1] - env.goal[1])
        scores[a] = r - 0.1 * dist_goal
    kept_actions = sorted(valid, key=lambda a: scores[a], reverse=True)[:n_keep]
    
    env_copy = GridWorldEnv(env.size, env.goal, env.trap)
    env_copy.pos = env.pos
    total_reward = 0
    
    for _ in range(max_steps):
        if not kept_actions: break
        action = random.choice(kept_actions)
        _, reward, done = env_copy.step(action)
        total_reward += reward
        if done: break
    return total_reward

def test_pruned():
    rewards = []
    for _ in range(100):
        env = GridWorldEnv(size=4)
        r = pruned_rollout(env, prune_ratio=0.5)
        rewards.append(r)
    print(f"剪枝50%平均回报: {np.mean(rewards):.3f}")
    return rewards

test_pruned()

print("\n第10章 蒙特卡洛树搜索 - 内容完成!")